# CC 도구 스마트 배치 — 병렬·단독 스케줄링 (GPT Responses API 재현)

클로드코드(CC)는 ReAct 도중 모델이 한 응답에 여러 tool_use를 내놓으면, 이를 **배치(batch)로 파티셔닝**해서 실행합니다.
핵심은 "분리"가 아니라 **단독(solo)** 개념입니다.

`partitionToolCalls`의 주석 원문 (`toolOrchestration.ts:86-90`):

```ts
/**
 * Partition tool calls into batches where each batch is either:
 * 1. A single non-read-only tool, or          ← unsafe = "단독 1개"
 * 2. Multiple consecutive read-only tools     ← safe = "연속"일 때만 병합
 */
```

핵심 규칙 4가지 (전부 소스 검증분):

| # | 규칙 | 근거 |
|---|---|---|
| 1 | **도구 단위 선언** — 각 도구가 스스로 `isConcurrencySafe` 선언. 파일 겹침 분석은 존재하지 않음 | `FileReadTool.ts:373`(Read true) · `Tool.ts:757-760`(기본값 false, "assume not safe") |
| 2 | **연속 safe만 병합, unsafe는 항상 단독** — unsafe끼리도(Edit, Write 연속이어도) 절대 안 뭉침 | `toolOrchestration.ts:109`(병합 조건) · `:112`(단독 push) |
| 3 | **배치 간 직렬, 배치 내 병렬** — 앞 배치가 전부 끝나야 다음 배치 시작. 동시성 한도 기본 10 | `toolOrchestration.ts:26-81` · `CLAUDE_CODE_MAX_TOOL_USE_CONCURRENCY` |
| 4 | **모델 emit 순서 그대로** — 재배열 없음. 먼저 끝나도 방출은 호출 순서, 짝짓기는 `tool_use_id` | `query.ts:820-824` · `StreamingToolExecutor.ts:412-439,435` |

이 노트북은 OpenAI **Responses API**의 멀티 펑션콜링(`parallel_tool_calls=True`) 위에 이 스케줄러를 그대로 재현합니다.
OpenAI API 자체는 병렬 호출을 **나열만** 할 뿐 실행 스케줄링은 전적으로 하네스(우리 코드) 몫 — CC도 동일하게 하네스 레벨에서 구현되어 있습니다.

In [1]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from types import SimpleNamespace

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # .env 의 OPENAI_API_KEY 로드

client = OpenAI()
MODEL = "gpt-5-nano"

## 1. 도구 단위 안전 선언 — `isConcurrencySafe`

CC 판정 방식 그대로 흉내냅니다:

- **Read/Grep 대응** (`read_file`, `grep_files`) → `isConcurrencySafe() { return true }` — 무조건 병렬 OK
- **Edit/Write 대응** (`edit_file`, `write_file`) → 선언 없음 → 기본값 `false` ("assume not safe") — 무조건 단독
- **Bash 대응** (`run_command`, 파티션 시연용) → **조건부**: 입력을 보고 "읽기 전용 명령인가"만 판정. 이웃 호출과의 파일 겹침은 안 봄

판정 함수가 입력(`inp`)을 받는 시그니처인 것도 CC와 동일 — Edit/Write는 `(_input?) => false`처럼 **입력을 받자마자 버립니다**.

In [2]:
READ_ONLY_COMMANDS = {"ls", "cat", "head", "tail", "pwd"}

TOOL_REGISTRY = {
    "read_file": {  # CC Read 대응
        "description": "파일 하나의 전체 내용을 읽는다.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "읽을 파일 경로, 예: /project/configs/app.ini"}},
            "required": ["path"],
            "additionalProperties": False,
        },
        "is_concurrency_safe": lambda inp: True,  # FileReadTool.ts:373
    },
    "grep_files": {  # CC Grep 대응
        "description": "모든 파일에서 pattern 이 포함된 줄을 찾는다.",
        "parameters": {
            "type": "object",
            "properties": {"pattern": {"type": "string", "description": "검색할 문자열"}},
            "required": ["pattern"],
            "additionalProperties": False,
        },
        "is_concurrency_safe": lambda inp: True,  # GrepTool.ts:183
    },
    "edit_file": {  # CC Edit 대응
        "description": "파일에서 old_string 을 new_string 으로 치환한다 (첫 1회).",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "수정할 파일 이름"},
                "old_string": {"type": "string", "description": "치환 대상 원문"},
                "new_string": {"type": "string", "description": "새 문자열"},
            },
            "required": ["path", "old_string", "new_string"],
            "additionalProperties": False,
        },
        "is_concurrency_safe": lambda inp: False,  # 기본값 false (Tool.ts:759 "assume not safe")
    },
    "write_file": {  # CC Write 대응
        "description": "파일을 새로 만들거나 전체 내용을 덮어쓴다.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "파일 이름"},
                "content": {"type": "string", "description": "파일 전체 내용"},
            },
            "required": ["path", "content"],
            "additionalProperties": False,
        },
        "is_concurrency_safe": lambda inp: False,  # 기본값 false
    },
    "run_command": {  # CC Bash 대응 — 파티션 판정 시연용 (API 에는 노출 안 함)
        "description": "셸 명령 실행 (시연용)",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
            "additionalProperties": False,
        },
        # 조건부 판정: "이 명령이 읽기 전용인가"만 본다 — 파일 겹침 분석 아님
        "is_concurrency_safe": lambda inp: inp.get("command", "").split()[0] in READ_ONLY_COMMANDS,
    },
}

FILE_TOOLS = ["read_file", "grep_files", "edit_file", "write_file"]

TOOLS = [
    {
        "type": "function",
        "name": name,
        "description": TOOL_REGISTRY[name]["description"],
        "parameters": TOOL_REGISTRY[name]["parameters"],
        "strict": True,
    }
    for name in FILE_TOOLS
]

## 2. 파티셔닝 — `partitionToolCalls` 재현

CC 원본 reduce(`toolOrchestration.ts:95-115`)의 번역입니다. 판정 실패는 전부 **보수적 false**:

- 인자 파싱(Zod `safeParse`) 실패 → `false`
- 판정 함수가 throw → `false`

병합 조건은 단 하나 — `내가 safe && 직전 배치도 safe`. unsafe는 첫 항에서 바로 탈락하므로 **직전 배치가 뭐든 항상 새 배치**(단독)가 됩니다.

In [3]:
def partition_tool_calls(function_calls):
    """CC partitionToolCalls (toolOrchestration.ts:95-115) 재현"""
    batches = []
    for call in function_calls:
        tool = TOOL_REGISTRY.get(call.name)
        try:
            parsed = json.loads(call.arguments)  # Zod safeParse 대응
            safe = bool(tool["is_concurrency_safe"](parsed)) if tool else False
        except Exception:
            safe = False  # 검증 실패 / 판정 중 throw → 보수적 false
        if safe and batches and batches[-1]["isConcurrencySafe"]:
            batches[-1]["blocks"].append(call)  # safe + 직전 배치도 safe → 합류 (:109)
        else:
            batches.append({"isConcurrencySafe": safe, "blocks": [call]})  # 아니면 무조건 새 배치 (:112)
    return batches


def brief(call):
    args = json.loads(call.arguments) if call.arguments else {}
    first = next(iter(args.values()), "")
    return f"{call.name}({str(first)[:24]})"


def print_partition(batches):
    parts = []
    for b in batches:
        tag = "🟢병렬" if b["isConcurrencySafe"] else "🔴단독"
        parts.append(tag + "[" + ", ".join(brief(c) for c in b["blocks"]) + "]")
    print("📦 파티션:", " → ".join(parts))

## 3. 정적 시연 — "분리"가 아니라 "단독"

API 호출 없이 파티션 로직만 검증합니다 (원문서 §05 시연 재현).

- **케이스 1**: `edit_file`·`write_file`이 연속인데도 한 unsafe 배치로 안 뭉치고 **각자 단독** ← 결정적 증거
- **케이스 2**: 중간에 `edit_file`이 끼면 앞의 `read_file`은 혼자 남음(1건짜리 병렬 배치) — **재배열 없음**
- **케이스 3**: 같은 도구라도 입력 따라 갈리는 조건부 판정 (CC Bash 대응)

In [4]:
_seq = 0

def fake(name, **args):
    global _seq
    _seq += 1
    return SimpleNamespace(name=name, arguments=json.dumps(args), call_id=f"tu_{_seq}")


print("케이스 1 — [read, read, grep, edit, write]")
print_partition(partition_tool_calls([
    fake("read_file", path="a.txt"),
    fake("read_file", path="b.txt"),
    fake("grep_files", pattern="foo"),
    fake("edit_file", path="a.txt", old_string="x", new_string="y"),
    fake("write_file", path="c.txt", content="hello"),
]))

print()
print("케이스 2 — [read, edit, read, grep]")
print_partition(partition_tool_calls([
    fake("read_file", path="a.txt"),
    fake("edit_file", path="b.txt", old_string="x", new_string="y"),
    fake("read_file", path="c.txt"),
    fake("grep_files", pattern="bar"),
]))

print()
print("케이스 3 — run_command 조건부 판정 (읽기 전용 명령만 safe)")
print_partition(partition_tool_calls([
    fake("run_command", command="ls -la"),
    fake("run_command", command="cat a.txt"),
    fake("run_command", command="rm old.txt"),
    fake("run_command", command="head b.txt"),
]))

케이스 1 — [read, read, grep, edit, write]
📦 파티션: 🟢병렬[read_file(a.txt), read_file(b.txt), grep_files(foo)] → 🔴단독[edit_file(a.txt)] → 🔴단독[write_file(c.txt)]

케이스 2 — [read, edit, read, grep]
📦 파티션: 🟢병렬[read_file(a.txt)] → 🔴단독[edit_file(b.txt)] → 🟢병렬[read_file(c.txt), grep_files(bar)]

케이스 3 — run_command 조건부 판정 (읽기 전용 명령만 safe)
📦 파티션: 🟢병렬[run_command(ls -la), run_command(cat a.txt)] → 🔴단독[run_command(rm old.txt)] → 🟢병렬[run_command(head b.txt)]


## 4. 목 파일시스템 + 도구 구현

병렬 효과가 타임라인에 보이도록 도구별 **모의 지연**을 넣습니다.
같은 병렬 배치라도 완료 순서가 뒤죽박죽이 되도록 지연을 서로 다르게 설정합니다.
실행이 실패해도 에러 문자열을 결과로 돌려줘야 모델이 스스로 복구할 수 있습니다.

In [5]:
from cc_mock_fs import FS as COMMON_FS

FILES = dict(COMMON_FS)  # 공통 목 코드베이스(orderhub, 40파일)의 사본 — 노트북에서 수정해도 원본 모듈은 유지


def tool_read_file(path):
    if path not in FILES:
        raise FileNotFoundError(f"'{path}' 없음. /project/ 로 시작하는 정확한 경로를 사용하세요.")
    return {"path": path, "content": FILES[path]}


def tool_grep_files(pattern):
    hits = [
        {"path": p, "line": line}
        for p, content in FILES.items()
        for line in content.splitlines()
        if pattern in line
    ]
    return {"pattern": pattern, "hits": hits}


def tool_edit_file(path, old_string, new_string):
    if path not in FILES:
        raise FileNotFoundError(f"'{path}' 없음")
    if old_string not in FILES[path]:
        raise ValueError(f"'{old_string}' 이 {path} 에 없음")
    FILES[path] = FILES[path].replace(old_string, new_string, 1)
    return {"path": path, "replaced": f"{old_string} → {new_string}"}


def tool_write_file(path, content):
    FILES[path] = content
    return {"path": path, "bytes": len(content.encode())}


TOOL_FUNCTIONS = {
    "read_file": tool_read_file,
    "grep_files": tool_grep_files,
    "edit_file": tool_edit_file,
    "write_file": tool_write_file,
}


def simulated_latency(name, args):
    """모의 지연(초) — read 두 건의 완료 순서가 착수 순서와 달라지게 설계"""
    if name == "read_file":
        return {"/project/configs/app.ini": 1.0, "/project/logs/app.log": 0.4}.get(args.get("path"), 0.5)
    return {"grep_files": 0.7, "edit_file": 0.6, "write_file": 0.6}.get(name, 0.5)

## 5. 배치 실행기 — 배치 간 직렬 · 배치 내 병렬 · 방출 순서 보존

CC `runTools`(`toolOrchestration.ts:26-81`) 구조 재현:

```
for 배치 of 파티션:            ← 바깥 루프 = 배치 간 "직렬" (앞 배치 완료까지 대기)
    if isConcurrencySafe: 배치 내 동시 실행 (한도 10)
    else:                 단독 실행
```

방출은 CC `StreamingToolExecutor`처럼 — **먼저 끝나도 방출은 emit 순서 고정**, 짝짓기는 위치가 아니라 `call_id`.

In [6]:
MAX_TOOL_USE_CONCURRENCY = 10  # CC: CLAUDE_CODE_MAX_TOOL_USE_CONCURRENCY 기본값


def run_one(call, t0):
    args = json.loads(call.arguments)
    started = time.perf_counter() - t0
    time.sleep(simulated_latency(call.name, args))  # 실제 I/O 대신 모의 지연
    try:
        output = json.dumps(TOOL_FUNCTIONS[call.name](**args), ensure_ascii=False)
    except Exception as e:
        output = f"Error: {e}"  # 에러도 tool 결과로 반환 → 모델이 스스로 복구
    ended = time.perf_counter() - t0
    return {"call": call, "output": output, "started": started, "ended": ended}


def execute_turn(function_calls):
    """한 턴의 function_call 전체를 CC 방식으로 스케줄링 실행"""
    batches = partition_tool_calls(function_calls)
    print_partition(batches)
    t0 = time.perf_counter()
    results_by_id = {}

    for i, batch in enumerate(batches, 1):
        blocks = batch["blocks"]
        if batch["isConcurrencySafe"]:
            print(f"  🟢 배치 {i} — CONCURRENT ({len(blocks)}건 동시 착수)")
            finish_order = []
            with ThreadPoolExecutor(max_workers=MAX_TOOL_USE_CONCURRENCY) as pool:
                futures = [pool.submit(run_one, c, t0) for c in blocks]
                for fut in as_completed(futures):
                    r = fut.result()
                    results_by_id[r["call"].call_id] = r
                    finish_order.append(brief(r["call"]))
            if len(blocks) > 1:
                print("     ⏱ 완료 순서(뒤죽박죽 가능):", " → ".join(finish_order))
        else:
            print(f"  🔴 배치 {i} — SERIAL (단독, 앞 배치 완료까지 대기)")
            for c in blocks:
                r = run_one(c, t0)
                results_by_id[r["call"].call_id] = r

    print("  ⏱ 타임라인 (턴 시작 기준):")
    for c in function_calls:
        r = results_by_id[c.call_id]
        print(f"     {brief(c):<36} {r['started']:.2f}s → {r['ended']:.2f}s")

    # 방출: 완료 순서와 무관하게 모델이 부른 순서(emit 순서) 그대로 — call_id 로 짝지음
    return [results_by_id[c.call_id] for c in function_calls]

## 6. 에이전트 루프 (Responses API)

`response.output` 전체를 `input`에 이어붙여 히스토리를 유지하고, 각 `function_call`마다
`function_call_output`(같은 `call_id`)을 **emit 순서대로** 추가합니다.

developer 지침으로 "독립적인 작업은 한 응답에 몰아서 호출하라"고 넛지해서,
한 턴에 safe·unsafe가 섞여 나오는 상황을 유도합니다.

In [7]:
DEVELOPER_PROMPT = (
    "너는 파일 작업 에이전트다. 파일은 반드시 도구 호출로만 다룬다.\n"
    "- 서로 독립적인 작업은 한 응답(한 턴)에 여러 function call 로 동시에 내놓아라.\n"
    "- 이미 주어진 정보로 인자를 채울 수 있는 호출은 뒤 턴으로 미루지 말고 같은 응답에 포함하라.\n"
    "- 모든 도구 호출이 끝나면 결과를 한국어로 간단히 요약하라."
)


def run_agent(user_message: str) -> str:
    input_list = [{"role": "user", "content": user_message}]
    turn = 0

    while True:
        turn += 1
        response = client.responses.create(
            model=MODEL,
            instructions=DEVELOPER_PROMPT,
            input=input_list,
            tools=TOOLS,
            parallel_tool_calls=True,
        )
        input_list += response.output  # function_call 포함 출력 전체를 히스토리에 보존

        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            return response.output_text

        print(f"\n{'=' * 64}")
        print(f"🔁 턴 {turn} — function_call {len(function_calls)}건 emit")
        results = execute_turn(function_calls)

        for r in results:  # emit 순서 그대로 function_call_output 추가
            input_list.append({
                "type": "function_call_output",
                "call_id": r["call"].call_id,
                "output": r["output"],
            })

## 7. 실행

읽기 3건(read×2 + grep)과 쓰기 2건(edit + write)이 섞인 요청입니다.
모델이 한 턴에 몰아서 emit하면 `🟢병렬[read, read, grep] → 🔴단독[edit] → 🔴단독[write]`
파티션이 그대로 관찰됩니다. (여러 턴으로 나눠 내놓아도 각 턴마다 파티션이 새로 계산됩니다.)

In [8]:
answer = run_agent(
    "/project/configs/app.ini 와 /project/logs/app.log 를 둘 다 읽고, "
    "전체 파일에서 'timeout' 이 들어간 줄도 검색해줘. "
    "그리고 /project/configs/app.ini 의 'timeout=30' 을 'timeout=60' 으로 수정한 다음, "
    "'결제 게이트웨이 타임아웃을 30초에서 60초로 상향' 이라는 내용으로 "
    "/project/docs/timeout_change.md 를 새로 만들어줘. "
    "수정할 원문(timeout=30)은 방금 내가 알려줬으니, 다섯 작업 모두 미루지 말고 "
    "지금 한 번의 응답에 전부 호출해도 된다."
)

print("\n=== 최종 답변 ===")
print(answer)

print("\n=== 변경/생성된 파일 상태 ===")
for p in ["/project/configs/app.ini", "/project/docs/timeout_change.md"]:
    print(f"📄 {p}: " + FILES.get(p, "(없음)").replace("\n", " ⏎ "))


🔁 턴 1 — function_call 5건 emit
📦 파티션: 🟢병렬[read_file(/project/configs/app.ini), read_file(/project/logs/app.log), grep_files(timeout)] → 🔴단독[edit_file(/project/configs/app.ini)] → 🔴단독[write_file(/project/docs/timeout_ch)]
  🟢 배치 1 — CONCURRENT (3건 동시 착수)


     ⏱ 완료 순서(뒤죽박죽 가능): read_file(/project/logs/app.log) → grep_files(timeout) → read_file(/project/configs/app.ini)
  🔴 배치 2 — SERIAL (단독, 앞 배치 완료까지 대기)


  🔴 배치 3 — SERIAL (단독, 앞 배치 완료까지 대기)


  ⏱ 타임라인 (턴 시작 기준):
     read_file(/project/configs/app.ini)  0.00s → 1.01s
     read_file(/project/logs/app.log)     0.00s → 0.40s
     grep_files(timeout)                  0.00s → 0.71s
     edit_file(/project/configs/app.ini)  1.01s → 1.61s
     write_file(/project/docs/timeout_ch) 1.61s → 2.22s



=== 최종 답변 ===
다음 다섯 작업을 한 번의 응답으로 처리했습니다.

1) /project/configs/app.ini 읽기
- 내용:
[server]
host=api.example.com
port=8000
timeout=30
retry=3

[database]
pool_size=10
pool_timeout=5
echo=false

[cache]
ttl_seconds=300
prefix=orderhub

2) /project/logs/app.log 읽기
- 내용:
[INFO] 2026-07-23 11:58:02 app — 서버 시작 (버전 0.4.2, DEBUG=True)
[INFO] 2026-07-23 12:00:11 app.orders — 주문 생성 user=2 total=15000
[warn] timeout exceeded at 12:01 — POST https://pay.example.com/v2/charge (30s)
[info] retry ok at 12:02 — 재시도 1회 만에 성공
[INFO] 2026-07-23 12:05:44 app.orders — 주문 생성 user=2 total=64000
[ERROR] 2026-07-23 12:07:19 app.payment — fetch failed after 3 retries (order=1042)
[INFO] 2026-07-23 12:07:19 app.orders — 주문 1042 상태 pending 유지, 수동 확인 필요
[warn] 2026-07-23 12:31:05 app.auth — 로그인 5회 연속 실패 username=hong

3) 전체 파일에서 'timeout' 이 들어간 줄 검색
- 검색 결과의 일부 (일부 예시 포함):
- /project/configs/app.ini: timeout=30
- /project/configs/app.ini: pool_timeout=5
- /project/src/app/database.py: engine = create_engine(settin

## 정리 — CC ↔ 이 노트북 대응표

| CC | 이 노트북 |
|---|---|
| 도구별 `isConcurrencySafe` 선언 (`Tool.ts:757-760` 기본값 false) | `TOOL_REGISTRY[*]["is_concurrency_safe"]` |
| `partitionToolCalls` reduce (`toolOrchestration.ts:95-115`) | `partition_tool_calls` |
| `runToolsConcurrently` / `runToolsSerially` (`:26-81`) | `ThreadPoolExecutor` / `for` 루프 |
| `CLAUDE_CODE_MAX_TOOL_USE_CONCURRENCY` 기본 10 | `MAX_TOOL_USE_CONCURRENCY = 10` |
| 방출은 받은 순서 고정 (`StreamingToolExecutor.ts:412-439`) | 결과를 emit 순서로 `function_call_output` 추가 |
| `tool_use_id` 매칭 (`:435`) | `call_id` 매칭 |

한 줄 요약: **배치 분할은 파일 충돌 분석이 아니라 도구 단위의 보수적 선언이다.**
unsafe 도구는 배치를 "가르는" 칸막이가 아니라 **자기 혼자만의 단독 배치**이며,
unsafe끼리도 절대 뭉치지 않고, 하네스는 모델이 내놓은 순서를 절대 재배열하지 않는다.
→ 따라서 **모델이 tool call을 내놓는 순서 자체가 병렬 효율을 좌우한다** — safe 호출을 앞쪽에 몰아 emit하는 게 유리한 이유.